# Align Standard He


## Notebook Flow


## 1. Locate the project and data


In [ ]:
from pathlib import Path as _Path
import os as _os
import sys as _sys
_candidates = (_Path.cwd(), *_Path.cwd().parents,
               _Path(globals().get("_code_root", _Path.cwd())))
_code_root = next(p for p in _candidates
                  if (p / 'xenium_pipeline/notebook_setup.py').is_file())
if str(_code_root) not in _sys.path:
    _sys.path.insert(0, str(_code_root))
from xenium_pipeline.notebook_setup import setup_notebook
_code_root, _project_root = setup_notebook()


In [ ]:
from pathlib import Path
import sys

DATASET_DIR_NAME = 'Janesick_2025'
if DATASET_DIR_NAME not in {'10x', 'Janesick_2023', 'Janesick_2025', 'Liu_2025'}:
    raise ValueError('Select a standard dataset; Bilous uses its landmark notebook.')
SAMPLE_NAMES = globals().get('SAMPLE_NAMES', None)  # None discovers every sample; or set ['Sample_A', 'Sample_B']
DATA_ROOT_OVERRIDE = globals().get('DATA_ROOT_OVERRIDE', None)

START_DIR = Path.cwd().resolve()
PROJECT_ROOT = _project_root

DATA_ROOT = Path(DATA_ROOT_OVERRIDE).expanduser().resolve() if DATA_ROOT_OVERRIDE else PROJECT_ROOT / DATASET_DIR_NAME
PIPELINE_DIR = _code_root
print('Project root:', PROJECT_ROOT)
print('Data root:', DATA_ROOT)


## 2. Check dependencies


In [ ]:
import importlib.util
import subprocess
from importlib.metadata import PackageNotFoundError, version

INSTALL_PACKAGES = False
REQUIREMENTS_PATH = PIPELINE_DIR / 'requirements.txt'
if not REQUIREMENTS_PATH.is_file():
    raise FileNotFoundError(f'Missing requirements file: {REQUIREMENTS_PATH}')

REQUIRED_MODULES = {
    'matplotlib': 'matplotlib',
    'numpy': 'numpy',
    'pandas': 'pandas',
    'Pillow': 'PIL',
    'pyarrow': 'pyarrow',
    'tifffile': 'tifffile',
    'tqdm': 'tqdm',
    'zarr': 'zarr',
    'imagecodecs': 'imagecodecs',
    'ipywidgets': 'ipywidgets',
}

if INSTALL_PACKAGES:
    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-r',
        str(REQUIREMENTS_PATH),
    ])

missing_modules = [
    package for package, module in REQUIRED_MODULES.items()
    if importlib.util.find_spec(module) is None
]
if missing_modules:
    raise ModuleNotFoundError(
        f'Missing modules: {missing_modules}. Set INSTALL_PACKAGES = True and rerun this cell.'
    )

zarr_version = version('zarr')
if int(zarr_version.split('.')[0]) >= 3:
    raise RuntimeError(
        f'zarr {zarr_version} is installed; this pipeline requires zarr>=2.16,<3.'
    )

print(f'Python {sys.version.split()[0]}')
print(f'Requirements: {REQUIREMENTS_PATH}')
for package in REQUIRED_MODULES:
    try:
        print(f'{package:12s} {version(package)}')
    except PackageNotFoundError:
        print(f'{package:12s} installed')


## 3. Load the dataset wrapper and discover samples


In [ ]:
import importlib

import pandas as pd
from IPython.display import Image as DisplayImage, clear_output, display
import ipywidgets as widgets

if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import xenium_pipeline.standard_xenium as shared_pipeline
importlib.reload(shared_pipeline)

from xenium_pipeline.dataset_config import get_dataset_config
pipeline = shared_pipeline
DATASET_CONFIG = get_dataset_config(DATASET_DIR_NAME)
print(f'Dataset config: {DATASET_CONFIG.display_name}')

samples = pipeline.discover_samples(DATA_ROOT, requested=SAMPLE_NAMES)
sample_names = [sample.name for sample in samples]
print(f'Discovered {len(sample_names)} sample(s):')
for name in sample_names:
    print(f'  - {name}')


### Inspect H&E and alignment sources


In [ ]:
he_rows = []
for sample in samples:
    alignment_csv = sample.alignment if sample.alignment and sample.alignment.is_file() else None
    if alignment_csv is not None:
        alignment_state = 'alignment_csv'
    elif DATASET_CONFIG.require_alignment:
        alignment_state = 'missing_alignment_csv'
    else:
        alignment_state = 'prealigned_or_alignment_not_required'
    he_rows.append({
        'sample': sample.name,
        'H&E OME-TIFF': bool(sample.he_image and sample.he_image.is_file()),
        'alignment_state': alignment_state,
        'alignment_csv': str(alignment_csv) if alignment_csv else '',
        'Xenium output dir': sample.outs.is_dir(),
        'Xenium output ZIP': bool(sample.outs_zip and sample.outs_zip.is_file()),
    })

he_inventory = pd.DataFrame(he_rows)
display(he_inventory)

missing_external_alignment = he_inventory.loc[
    he_inventory['alignment_state'].eq('missing_alignment_csv')
]
if not missing_external_alignment.empty:
    print('Samples missing an alignment CSV:')
    display(missing_external_alignment[['sample', 'alignment_state']])


## 4. Validate inputs before alignment QC


In [ ]:
validation, _ = pipeline.run_all_samples(
    DATA_ROOT,
    **(DATASET_CONFIG.standard_run_kwargs() | dict(sample_names=sample_names, validate_only=True, skip_processed=False, require_alignment=False)),
)
display(validation)

invalid = validation.loc[validation['status'] != 'valid']
if not invalid.empty:
    details = invalid[['sample', 'error']].to_string(index=False)
    raise RuntimeError(f'Fix invalid non-alignment inputs before alignment import/QC:\n{details}')

print('All selected samples passed non-alignment input validation.')


## 5. Import missing image alignment CSVs from Xenium Explorer


In [ ]:
missing_alignment_samples = pipeline.samples_missing_imagealignment(
    DATA_ROOT,
    sample_names=sample_names,
)
print('Samples missing `_he_imagealignment.csv`:')
for name in missing_alignment_samples:
    print(f'  - {name}')
if not missing_alignment_samples:
    print('  (none)')


### 5a. Export alignment files from Xenium Explorer


In [ ]:
print('Xenium Explorer image-alignment instructions:')
print(pipeline.XENIUM_EXPLORER_ALIGNMENT_HELP_URL)
print()
print('Place each downloaded alignment zip here:')
names_to_show = missing_alignment_samples if missing_alignment_samples else list(sample_names or [])
for name in names_to_show:
    print(f'  - {DATA_ROOT / name}')
if not names_to_show:
    print('  (no selected samples are currently missing an alignment CSV)')
if not missing_alignment_samples:
    print('No missing alignment CSVs were detected. To replace an existing CSV, place the zip in the sample folder and keep OVERWRITE_ALIGNMENT_CSV=True below.')


### 5b. Import downloaded Xenium Explorer zips


In [ ]:
IMPORT_XENIUM_EXPLORER_ZIPS = True
OVERWRITE_ALIGNMENT_CSV = True

if IMPORT_XENIUM_EXPLORER_ZIPS:
    alignment_import_summary = pipeline.import_xenium_explorer_alignment_zips(
        DATA_ROOT,
        sample_names=sample_names,
        overwrite=OVERWRITE_ALIGNMENT_CSV,
    )
    display(alignment_import_summary)

    failed_imports = alignment_import_summary.loc[
        alignment_import_summary['status'].eq('failed')
    ]
    if not failed_imports.empty:
        details = failed_imports[['sample', 'zip', 'error']].to_string(index=False)
        raise RuntimeError(f'Fix failed alignment imports before QC: {details}')
else:
    print('Set IMPORT_XENIUM_EXPLORER_ZIPS=True after placing downloaded zips in sample folders.')


### 5c. Confirm active alignment CSVs


In [ ]:
missing_alignment_samples = pipeline.samples_missing_imagealignment(
    DATA_ROOT,
    sample_names=sample_names,
)
print('Remaining samples missing `_he_imagealignment.csv`:')
for name in missing_alignment_samples:
    print(f'  - {name}')
if not missing_alignment_samples:
    print('  (none)')
else:
    print('Xenium Explorer instructions:', pipeline.XENIUM_EXPLORER_ALIGNMENT_HELP_URL)


## 6. Generate and review alignment QC


In [ ]:
alignment_validation_kwargs = DATASET_CONFIG.validation_kwargs()
alignment_qc_summary = pipeline.run_alignment_qc(
    DATA_ROOT,
    sample_names=sample_names,
    allow_manual_alignment=False,
    fail_fast=True,
    **alignment_validation_kwargs,
)
display(alignment_qc_summary)

alignment_failures = alignment_qc_summary.loc[
    alignment_qc_summary['status'] != 'success'
]
if not alignment_failures.empty:
    details = alignment_failures[['sample', 'error']].to_string(index=False)
    raise RuntimeError(f'Fix alignment failures before tile creation: {details}')

print('Alignment QC images are ready. Inspect every sample before running create_tiles.')


### Check alignment QC images


In [ ]:
alignment_qc_sample_names = list(sample_names)
if not alignment_qc_sample_names:
    raise ValueError('No samples are available for alignment QC review.')

alignment_dropdown = widgets.Dropdown(
    options=alignment_qc_sample_names,
    value=alignment_qc_sample_names[0],
    description='Sample',
    layout=widgets.Layout(width='520px'),
)
alignment_previous = widgets.Button(description='Previous', icon='chevron-left')
alignment_next = widgets.Button(description='Next', icon='chevron-right')
alignment_output = widgets.Output()


def show_alignment_sample(sample_name):
    sample_index = alignment_qc_sample_names.index(sample_name) + 1
    plot_path = DATA_ROOT / sample_name / f'{sample_name}_xenium_he_alignment.png'
    with alignment_output:
        clear_output(wait=True)
        print(f'Alignment QC sample {sample_index} / {len(alignment_qc_sample_names)}: {sample_name}')
        print(plot_path)
        if plot_path.is_file():
            display(DisplayImage(filename=str(plot_path)))
        else:
            print('Missing. Run the alignment QC cell above first.')


def shift_alignment_sample(offset):
    current_index = alignment_qc_sample_names.index(alignment_dropdown.value)
    alignment_dropdown.value = alignment_qc_sample_names[(current_index + offset) % len(alignment_qc_sample_names)]


def on_alignment_sample_change(change):
    if change['name'] == 'value' and change['new']:
        show_alignment_sample(change['new'])


alignment_previous.on_click(lambda _: shift_alignment_sample(-1))
alignment_next.on_click(lambda _: shift_alignment_sample(1))
alignment_dropdown.observe(on_alignment_sample_change, names='value')

display(widgets.HBox([alignment_previous, alignment_dropdown, alignment_next]))
display(alignment_output)
show_alignment_sample(alignment_dropdown.value)


## 7. Write tile input manifest


In [ ]:
TILE_INPUT_MANIFEST = DATA_ROOT / 'tile_input_manifest.csv'
tile_input_manifest = pipeline.write_tile_input_manifest(
    DATA_ROOT,
    sample_names=sample_names,
    output_path=TILE_INPUT_MANIFEST,
)
display(tile_input_manifest)
print('Wrote tile input manifest:', TILE_INPUT_MANIFEST)


## 8. Prepare-stage outputs and handoff
